In [ ]:
import os
import hashlib
import shutil
from pathlib import Path

f_path = "/home/fisaa/AIRE410_Dataset/archive/OCT2017"
def md5(file_path):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

In [ ]:

def remove_duplicates_train_only(root_dir):
    root = Path(root_dir)

    train_dir = root / "train"
    if not train_dir.exists():
        print("Error: 'train' folder not found.")
        return

    # New location for duplicates INSIDE OCT2017 folder
    dup_root = root / "train_duplicates"

    hash_map = {}

    print("Scanning TRAIN folder only...\n")

    for img_path in train_dir.rglob("*.*"):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".tiff"]:
            continue

        file_hash = md5(img_path)

        if file_hash not in hash_map:
            hash_map[file_hash] = img_path
            continue

        # Duplicate found → preserve class folder structure
        rel_class = img_path.parent.name  # CNV / DME / DRUSEN / NORMAL
        target_dir = dup_root / rel_class
        target_dir.mkdir(parents=True, exist_ok=True)

        dest_path = target_dir / img_path.name

        print(f"Duplicate → {img_path}  →  {dest_path}")

        shutil.move(str(img_path), str(dest_path))

    print("\n Duplicate removal complete.")
    print(f" Duplicates saved in: {dup_root}")


if __name__ == "__main__":
    remove_duplicates_train_only(f_path)